[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aditya-siraskar/PINN_FTRE/blob/main/notebooks/07_phase4_stage6_real_data.ipynb)

# 07 · Stage 6: real data, non-linear portfolio and backtests

**Phase:** Phase 4: empirical

**Goal:** Rolling one-day-ahead VaR/ES on a NIFTY basket and an S&P sector-ETF basket (2005–), including an index-option position; benchmarks and backtests (RQ2).

**Outputs:** `data/raw/`, `figures/stage6_empirical/`

> Runtime → Change runtime type → **T4 GPU**. Run the setup cell first. With `USE_DRIVE = True` the repo lives in
> `MyDrive/PINN_FTRE`, so figures and results persist. Core logic lives in `pinn_tailrisk/` and `scripts/`;
> this notebook only runs it and shows the results.

In [ ]:
# @title Setup: repo on Google Drive, install, GPU check  { display-mode: "form" }
USE_DRIVE = True   # keep figures/results across Colab sessions (recommended)
REPO_URL = "https://github.com/aditya-siraskar/PINN_FTRE"

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
else:
    BASE = "/content"
ROOT = f"{BASE}/PINN_FTRE"
if not os.path.exists(ROOT):
    !git clone -q {REPO_URL} "{ROOT}"
else:
    rc = os.system(f'git -C "{ROOT}" pull -q --ff-only')
    if rc != 0:   # usually: result files on Drive that have since been committed to GitHub
        print("WARNING: git pull failed; the Drive copy is out of date. To sync (keeps untracked files):
"
              f'  !git -C "{ROOT}" fetch -q origin && git -C "{ROOT}" reset -q --hard origin/main')
%cd "{ROOT}"
!pip install -q -e .
sys.path.insert(0, ROOT)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (Runtime > Change runtime type > T4 GPU)")

In [ ]:
# autoreload picks up library edits without a restart. Colab's IPython build imports the
# removed `imp` module on Python >= 3.12, so it is optional: if it is skipped, use
# Runtime > Restart session after editing pinn_tailrisk/.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload unavailable here; restart the session after editing the library")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pinn_tailrisk import plotting as P
from pinn_tailrisk.notebook import show_figures, load_metrics
P.use_style()

## Plan for this stage
**To build:** `pinn_tailrisk/data.py` (download and cache), `calibrate.py` (rolling maximum-likelihood fit of the Merton mixture), `benchmarks.py` (Gaussian, HS, FHS, GARCH-t via `arch`, DCC-GARCH-t, EVT-POT), `backtest.py` (Kupiec, Christoffersen, FZ0 score, Diebold–Mariano / MCS).

Split: develop 2005–2017, out-of-sample 2018– (COVID included). No look-ahead: refit using past data only.

**Planned figures:**
- fig01 return series and rolling vol
- fig02 tail log-histogram vs normal
- fig03 rolling calibrated parameters
- fig04 VaR/ES forecasts with breaches
- fig05 backtest table and score comparison

## Preview with the current library

In [ ]:
# Data preview: download and cache daily closes, then the first EDA figure
import yfinance as yf, pathlib
TICKERS = {"NIFTY": ["^NSEI", "RELIANCE.NS", "HDFCBANK.NS", "ICICIBANK.NS", "INFY.NS", "TCS.NS"],
           "SP500": ["^GSPC", "XLK", "XLF", "XLE", "XLV", "XLY"]}
pathlib.Path("data/raw").mkdir(parents=True, exist_ok=True)
for name, tick in TICKERS.items():
    f = pathlib.Path(f"data/raw/{name}.csv")
    if not f.exists():
        yf.download(tick, start="2005-01-01", auto_adjust=True, progress=False)["Close"].to_csv(f)
px = pd.read_csv("data/raw/NIFTY.csv", index_col=0, parse_dates=True)
r = np.log(px["^NSEI"]).diff().dropna()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3))
axes[0].plot(r.index, r * 100, color=P.INK_2, lw=0.5); axes[0].set(title="NIFTY 50 daily log return", ylabel="%")
z = (r - r.mean()) / r.std(); bins = np.linspace(-10, 6, 80)
h, e = np.histogram(z, bins, density=True); c = 0.5 * (e[1:] + e[:-1])
axes[1].semilogy(c[h > 0], h[h > 0], lw=0, marker="o", ms=3.5, color=P.INK, label="NIFTY 50")
from scipy import stats
axes[1].semilogy(c, stats.norm.pdf(c), **P.style("gaussian")); axes[1].set(title="Standardised returns vs Gaussian", xlabel="z")
axes[1].legend(); P.panel_labels(axes); fig.tight_layout()
P.savefig(fig, "stage6_empirical", "fig01_nifty_eda"); plt.show()

## Run the stage
_Placeholder: the stage script is not written yet._

In [ ]:
# !python scripts/stage6_real_data.py
# show_figures("...")